## Gold Layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Read Silver Table

In [0]:
df_silver = spark.read.table("fintech.silver.stock_prices")

### Update dim_stock
- For dim_stock we create stock_key as a surrogate key incremental by using row_number()

In [0]:
# =========================================================
# Get distinct symbols from Silver
# =========================================================
df_symbols = (
    df_silver
    .select("symbol")
    .distinct()
    .filter(F.col("symbol").isNotNull())
)

# =========================================================
# Check if dim_stock exists
# =========================================================
dim_stocks_exists = spark.catalog.tableExists(
    "fintech.gold.dim_stock"
)

# =========================================================
# Create dimension if it does not exist
# =========================================================
if not dim_stocks_exists:

    window = Window.orderBy("symbol")

    df_dim_stock = (
        df_symbols.withColumn(
            "stock_key", 
            F.row_number().over(window) 
        )
        .select(
            "stock_key", 
            "symbol"
        )
    )

    (
        df_dim_stock
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("fintech.gold.dim_stock")
    )

# =========================================================
# Existing dimension
# =========================================================
else:
    df_dim_stock = (
        spark.table("fintech.gold.dim_stock")
        .select("stock_key", "symbol")
    )

    # =====================================================
    # Find new stocks
    # =====================================================
    df_new_symbols = (
        df_symbols
        .join(
            df_dim_stock.select("symbol"),
            on="symbol",
            how="left_anti"
        )
    )

    # =====================================================
    # Generate new surrogate keys
    # =====================================================
    max_key = (
        df_dim_stock
        .agg(
            F.max("stock_key").alias("max_key")
        )
        .collect()[0]["max_key"]
    )

    if max_key is None:
        max_key = 0

    window = Window.orderBy("symbol")

    df_new_symbols = (
        df_new_symbols
        .withColumn(
            "stock_key",
            F.row_number().over(window) + F.lit(max_key)
        )
        .select("stock_key", "symbol")
    )

    # =====================================================
    # Insert new stocks
    # =====================================================
    if not df_new_symbols.isEmpty():
        (
            df_new_symbols
            .write
            .format("delta")
            .mode("append")
            .saveAsTable("fintech.gold.dim_stock")
        )

### Update dim_date
- For dim_date we create date_key as a surrogate key in more deterministic way by using date_format(YYYYMMDD)

In [0]:
# =========================================================
# Get distinct dates from Silver
# =========================================================
df_dates = (
    df_silver
    .select("trade_date")
    .distinct()
    .filter(
        F.col("trade_date").isNotNull()
    )
)

# =========================================================
# Check if dim_date exists
# =========================================================
dim_date_exists = spark.catalog.tableExists(
    "fintech.gold.dim_date"
)

# =========================================================
# Create dimension if it does not exist
# =========================================================
if not dim_date_exists:

    df_dim_date = (
        df_dates
        .withColumn(
            "date_key",
            F.date_format(
                "trade_date",
                "yyyyMMdd"
            ).cast("int")
        )
        .withColumn(
            "full_date",
            F.col("trade_date")
        )
        .withColumn(
            "year",
            F.year("trade_date")
        )
        .withColumn(
            "quarter",
            F.quarter("trade_date")
        )
        .withColumn(
            "month",
            F.month("trade_date")
        )
        .withColumn(
            "month_name",
            F.date_format(
                "trade_date",
                "MMMM"
            )
        )
        .withColumn(
            "week_of_year",
            F.weekofyear("trade_date")
        )
        .withColumn(
            "day",
            F.dayofmonth("trade_date")
        )
        .withColumn(
            "day_of_week",
            F.dayofweek("trade_date")
        )
        .withColumn(
            "day_name",
            F.date_format(
                "trade_date",
                "EEEE"
            )
        )
        .withColumn(
            "is_weekend",
            F.dayofweek("trade_date").isin(1, 7)
        )
        .select(
            "date_key",
            "full_date",
            "year",
            "quarter",
            "month",
            "month_name",
            "week_of_year",
            "day",
            "day_of_week",
            "day_name",
            "is_weekend"
        )
    )

    (
        df_dim_date
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("fintech.gold.dim_date")
    )

# =========================================================
# Existing dimension
# =========================================================
else:
    df_dim_date = (
        spark.table("fintech.gold.dim_date")
    )

    # =====================================================
    # Find new dates
    # =====================================================
    df_new_dates = (
        df_dates
        .join(
            df_dim_date.select("full_date"),
            on=df_dates.trade_date == df_dim_date.full_date,
            how="left_anti"
        )
        .select("trade_date")
    )

    # =====================================================
    # Generate attributes for new dates
    # =====================================================
    df_new_dates = (
        df_new_dates
        .withColumn(
            "date_key",
            F.date_format(
                "trade_date",
                "yyyyMMdd"
            ).cast("int")
        )
        .withColumn(
            "full_date",
            F.col("trade_date")
        )
        .withColumn(
            "year",
            F.year("trade_date")
        )
        .withColumn(
            "quarter",
            F.quarter("trade_date")
        )
        .withColumn(
            "month",
            F.month("trade_date")
        )
        .withColumn(
            "month_name",
            F.date_format(
                "trade_date",
                "MMMM"
            )
        )
        .withColumn(
            "week_of_year",
            F.weekofyear("trade_date")
        )
        .withColumn(
            "day",
            F.dayofmonth("trade_date")
        )
        .withColumn(
            "day_of_week",
            F.dayofweek("trade_date")
        )
        .withColumn(
            "day_name",
            F.date_format(
                "trade_date",
                "EEEE"
            )
        )
        .withColumn(
            "is_weekend",
            F.dayofweek("trade_date").isin(1, 7)
        )
        .select(
            "date_key",
            "full_date",
            "year",
            "quarter",
            "month",
            "month_name",
            "week_of_year",
            "day",
            "day_of_week",
            "day_name",
            "is_weekend"
        )
    )
    # =====================================================
    # Insert new dates
    # =====================================================
    if not df_new_dates.isEmpty():
        (
            df_new_dates
            .write
            .format("delta")
            .mode("append")
            .saveAsTable("fintech.gold.dim_date")
        )

### Update fact_stock_prices

In [0]:
# =========================================================
# Read Gold dimensions
# =========================================================
df_dim_stock = spark.table(
    "fintech.gold.dim_stock"
)

df_dim_dat = spark.table(
    "fintech.gold.dim_date"
)

# =========================================================
# Build fact dataset
# =========================================================
df_fact_stock_prices = (
    df_silver.alias("p")

    .join(
        df_dim_stock.alias("s"),
        on= F.col("p.symbol") == F.col("s.symbol"),
        how="inner"
    )

    .join(
        df_dim_date.alias("d"),
        on = F.col("p.trade_date") == F.col("d.full_date"),
        how="inner"
    )

    .select(
        F.col("s.stock_key"),
        F.col("d.date_key"),

        F.col("p.open"),
        F.col("p.high"),
        F.col("p.low"),
        F.col("p.close"),
        F.col("p.volume"),
        F.col("p.change"),
        F.col("p.change_percent"),
        F.col("p.vwap"),

        F.col("p._ingestion_timestamp"),
        F.col("p._source_file")

    )
)

# =========================================================
# Check if fact table exists
# =========================================================
fact_exists = spark.catalog.tableExists(
    "fintech.gold.fact_stock_prices"
)

# =========================================================
# Create fact table if it does not exist
# =========================================================
if not fact_exists:

    (
        df_fact_stock_prices
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("fintech.gold.fact_stock_prices")
    )

# =========================================================
# Incremental MERGE
# =========================================================
else:
    fact_table = DeltaTable.forName(
        spark, "fintech.gold.fact_stock_prices"
    )

    (
        fact_table.alias("target")
        .merge(
            df_fact_stock_prices.alias("source"),
            """
                target.stock_key = source.stock_key
                AND target.date_key = source.date_key
            """
        )
        .whenMatchedUpdate(
            condition="""
                source._ingestion_timestamp > target._ingestion_timestamp
            """,
            set={
                "open": "source.open",
                "high": "source.high",
                "low": "source.low",
                "close": "source.close",
                "volume": "source.volume",
                "change": "source.change",
                "change_percent": "source.change_percent",
                "vwap": "source.vwap",
                "_ingestion_timestamp":
                    "source._ingestion_timestamp",
                "_source_file":
                    "source._source_file"
            }
        )

        .whenNotMatchedInsertAll()
        
        .execute()
    )